---
title: "Exercise 1A: SpatialExperiment - VisiumHD binned"
format: html
editor: source
editor_options:
  chunk_output_type: console
jupyter: ir
---

## Learning Objectives

By the end of this exercise, you will be able to:

-   Understand the structure of a `SpatialExperiment` object.
-   Access and interpret `spatialCoords` and `imgData`.
-   Inspect assay, row, and column metadata.
-   Perform basic subsetting operations on `SpatialExperiment` objects.
-   Visualize Visium HD binned data on the tissue image.

## Libraries

In [ ]:
#| echo: false
#| warning: false
#| output: false

library(downlit)
library(xml2)
library(GenomeInfoDb)
library(scrapper)
# library(SingleR)
# library(BayesSpace)
library(xgboost)
library(namer)
library(styler)
library(yaml)

In [ ]:
#| warning: false
#| output: false

library(SpatialExperiment)
library(VisiumIO)
library(ggspavis)
library(patchwork)
library(dplyr)
library(scuttle)
library(HDF5Array)

<!-- TO DO: check here and in other qmd which packages are required -->

## Data for the course

We will work on a human colorectal cancer study by [Oliveira _et al_](https://www.nature.com/articles/s41588-025-02193-3). The dataset contains normal adjacent tissue (NAT) and colorectal carcinoma (CRC) from 5 patients. We will focus on data from a patient called "P2", available from the [10X website](https://www.10xgenomics.com/datasets/visium-hd-cytassist-gene-expression-libraries-of-human-crc). 

For this sample, the study provides Visium HD data at 2, 8, and 16 um resolution and the per-cell output after cell segmentation. Matching Xenium data generated from serial sections of the same sample is also available and can be used for cross-platform comparisons.

In this exercise we will work with **Visium HD** data and focus on the "binned output" available as an output of the Space Ranger pipeline (version 3.0). The full output is very large, so for practical reasons we use a region of interest, selected to include an interesting part of the tissue, because it contains gland-like epithelial structures together with stromal or lower-density areas:


In [ ]:
# Define region of interest (roi)
roi_visium <- c(
  xmin = 49000,
  xmax = 58000,
  ymin = 7500,
  ymax = 16000
)
roi_visium

*Note*: In `Exercise 1B` we focus on the matching Xenium dataset from a serial section of the same P2CRC sample, subsetted to an approximately matching region of interest.


Download P2 Visium HD data:

In [ ]:
options(timeout = 600)

data_dir <- "data"
dir.create(data_dir, showWarnings = FALSE)

folders <- c(
  "day3",
  "FLEX",
  "Human_Colon_Cancer_P2",
  "segmented_outputs"
)

local_paths <- file.path(data_dir, "Human_Colon_Cancer_P2")

# CASE 1: already present locally (downloaded OR symlinked)
if (file.exists(local_paths)) {

  message("Data already available in ./data — nothing to do")

# CASE 2: available in /data → create symlinks
} else if (all(file.exists(file.path("/data", folders)))) {

  message("Data found in /data — creating symlinks")

  for (f in folders) {
    src <- file.path("/data", f)
    dest <- file.path(data_dir, f)

    if (!file.exists(dest)) {
      file.symlink(from = src, to = dest)
    }
  }

# CASE 3: missing → download
} else {

  message("Data not found — downloading into ./data")

  download.file(
    url = "https://intro-spatial-transcriptomics-training.s3.eu-central-1.amazonaws.com/spatial_course_data.tar.gz",
    destfile = file.path(data_dir, "spatial_course_data.tar.gz"),
    mode = "wb"
  )

  untar(
    tarfile = file.path(data_dir, "spatial_course_data.tar.gz"),
    exdir = data_dir
  )

  file.remove(file.path(data_dir, "spatial_course_data.tar.gz"))
}

We will first import the 16 um binned Visium HD output into a `SpatialExperiment` object:

In [ ]:
# Import data into a SpatialExperiment object
spe <- TENxVisiumHD(
  spacerangerOut = "data/Human_Colon_Cancer_P2/",
  processing = "filtered",
  format = "h5",
  images = "lowres",
  bin_size = "016"
) |>
  import()

# Use unique Symbol for rownames
rownames(spe) <- uniquifyFeatureNames(
  ID = rowData(spe)$ID,
  names = rowData(spe)$Symbol
)

spe

Then, we will subset it to the region of interest. The coordinates are in the native Visium HD image coordinate system returned by `spatialCoords()`.

In [ ]:
# Explore the spatial coordinates
head(spatialCoords(spe))

# Subset to the region of interest based on spatial coordinates
spe <- spe[, spatialCoords(spe)[, 1] >= roi_visium[["xmin"]] &
  spatialCoords(spe)[, 1] <= roi_visium[["xmax"]] &
  spatialCoords(spe)[, 2] >= roi_visium[["ymin"]] &
  spatialCoords(spe)[, 2] <= roi_visium[["ymax"]]]

spe

::: callout-important
## Exercise 1

Have a look at the paper and the 10X Genomics website.

-   How are transcript molecules measured in Visium HD?
-   What does the 16 um binned output represent?
-   Why might it be a good idea to use 16 um bins instead of the smallest 2 um bins? What might be the issues?
-   Which report from the Space Ranger tool can you inspect to get an overview of sample and processing quality?
:::

::: {.callout-tip collapse="true"}
## Answer

The measurements of RNA molecules are indirect: targeted probe pairs are hybridized, ligated, captured on the Visium HD slide, and sequenced. The binned Visium HD outputs aggregate molecules over square spatial bins.

The smallest Visium HD bins are 2x2 um. Larger bins are pools of neighboring 2 um bins: an 8 um bin aggregates 16 smaller bins, and a 16 um bin aggregates 64 smaller bins. This increases the number of UMIs per gene and per bin and reduces the sparsity of the data. This is of course at the cost of spatial resolution, and it should be kept in mind that a 16 um may captures material from multiple cells. For this course, 16 um bins are a practical compromise. They make the object smaller and faster to plot and process while still preserving tissue-level spatial structure. 

The [Space Ranger web summary](https://www.10xgenomics.com/datasets/visium-hd-cytassist-gene-expression-libraries-of-human-crc) and metrics summary are useful reports for checking processing quality before using the sample in the course.
:::

## Exploring the object

In this exercise, we will inspect the `SpatialExperiment` class used to store the data. `SpatialExperiment` extends `SingleCellExperiment`, classically used for scRNA-seq analysis, so familiar accessors such as `colData()`, `rowData()`, `assay()`, and `reducedDims()` still apply.

![Overview of the `SpatialExperiment` class structure](../assets/images/spe.png)

*Note*: The practical `Exercise 1B` uses a Xenium dataset stored into a `SpatialFeatureExperiment` class, which is an extension of the `SpatialExperiment` incorporating geometries, allowing to store cell segmentation polygons for example.


::: callout-important
## Exercise 2

Check the outputs of:

-   `colData()`
-   `rowData()`
-   `assay()`
-   `reducedDims()`

Then check the spatial-specific accessors:

-   `spatialCoords()`
-   `imgData()`
-   `imgRaster()`

Questions:

-   What kind of data is stored in each slot?
-   How many bins and genes are retained in the subsetted `spe` object?
-   Are all retained bins within the tissue area?
-   Bonus: what class is used to store the UMI count matrix? What is special about it?
:::

::: {.callout-tip collapse="true"}
## Answer

The slot `colData` contains metadata for each spatial bin:

In [ ]:
colData(spe) |> head()
ncol(spe)

We can check whether bins are covered by tissue:

In [ ]:
colData(spe) |>
  as.data.frame() |>
  group_by(in_tissue) |>
  summarise(number = n())

<!-- TO DO: This doesn't seem to work very well since some spots obviously are not in_tissue...-->

The slot `rowData` contains gene metadata:

In [ ]:
rowData(spe) |> head()
nrow(spe)

The assay stores the UMI counts matrix:

In [ ]:
assay(spe)
class(assay(spe))

The `DelayedArray` class allows to store out-of-memory representations of count matrices, which are saved as `.h5` files on the disk. This is very useful for efficient processing of large datasets (100,000s of cells or spots)

The `reducedDims` slot is empty at this stage:

In [ ]:
reducedDims(spe)

The `spatialCoords` slot stores image coordinates for each bin:

In [ ]:
spatialCoords(spe) |> head()

The `imgData` slot lists the linked tissue image and scale factor:

In [ ]:
imgData(spe)

The image can be exported as a raster:

In [ ]:
imgRaster(spe) |> plot()

:::

## Visualizing the tissue region

To get a visual overview, we use `plotVisium()` function from the `ggspavis` package. First we plot the tissue image without bins:

In [ ]:
plotVisium(spe, spots = FALSE)

Then we overlay the 16 um bins to see which part of the tissue was covered by the array:

In [ ]:
plotVisium(spe, point_shape = 22, point_size = 0.5)

::: callout-important
## Exercise 3

Why do we see only a specific region of the slide?
:::

::: {.callout-tip collapse="true"}
## Answer

The full Visium HD output is much larger than needed for this introductory exercise. At the start of this exercise, we subsetted the object to one representative tissue region so that it remains fast to plot and process.
:::

It is possible to colour bins according to a gene or a column in `colData`, for example `PIGR`. We renamed the rows to gene symbols after import, which is why we can refer to this marker as `"PIGR"` rather than by its Ensembl ID. At this stage we plot the number of UMIs since log-normalized counts will be introduced later in the course.

In [ ]:
#| warning: false

plotVisium(spe,
  annotate = "PIGR",
  assay = "counts",
  zoom = TRUE,
  point_size = 1,
  point_shape = 22
)

::: callout-important
## Exercise 4

Check the usage of `plotVisium` with `?plotVisium`. Create a plot that zooms into the selected region and colors the bins according to column `array_row`.

In which slot is `array_row` stored, and what does it represent?
:::

::: {.callout-tip collapse="true"}
## Answer

The column `array_row` is stored in `colData`. It represents the row position of each bin in the Visium HD grid, so the plot shows a smooth coordinate gradient across the selected region. This is technical spatial metadata, not a gene expression pattern.

In [ ]:
plotVisium(
  spe,
  annotate = "array_row",
  zoom = TRUE,
  point_shape = 22
)

:::

## Retain only observations that map to tissue

::: callout-important
## Exercise 5

Subset the `SpatialExperiment` object to retain only bins that are within the tissue area.

How many bins were present in `spe`, and how many are left after selecting bins overlapping the tissue?
:::

::: {.callout-tip collapse="true"}
## Answer

In [ ]:
table(colData(spe)$in_tissue) 
## all spots, although it's clear that some do not overlap the tissue...

## let's assume that the spots not overlapping the tissue will not collect many counts
quantile(colSums(counts(spe)), prob=0.01)
quantile(colSums(counts(spe)), prob=0.02)
sub <- colSums(counts(spe)) > 50
table(sub)

plotVisium(
  spe[, sub],
  annotate = "array_row",
  zoom = TRUE,
  point_shape = 22
)

Filtering on the library size thus seems to remove a few"empty" bins not overlapping the tissue. Let's keep this in mind for the Exercise 2 where we perform QC filtering!
:::

## Save the object

We save the filtered `SpatialExperiment` object for the next steps. Remember that the count matrix is stored as an out-of-memory `DelayedArray` object, so the object cannot be saved with the usual `saveRDS` command. The following command saves the assay matrices as `.h5` files along with the `SpatialExperiment` object as an `.rds` file.

In [ ]:
dir.create("results/day1", showWarnings = FALSE, recursive = TRUE)
saveHDF5SummarizedExperiment(spe,
  dir = "results/day1", prefix = "01.1a_spe_", replace = TRUE,
  chunkdim = NULL, level = NULL, as.sparse = NA,
  verbose = NA
)

Clear your environment:

In [ ]:
#| eval: false
rm(list = ls())
gc()
.rs.restartR()

::: callout-important
**Key Takeaways:**

-   `SpatialExperiment` is a convenient container for spot- or bin-level spatial transcriptomics data.
-   Visium HD binned outputs can be represented naturally as a `SpatialExperiment`.
-   Course subsets are chosen to balance biological realism with practical runtime.
:::